<img src="https://fedesarrollo.org.co/logo-Fede.png" alt="Fedesarrollo" style="float: right; width: 300px; height: auto;">
<img src="https://upload.wikimedia.org/wikipedia/commons/f/f9/Logo_BID_Espa%C3%B1ol.png" alt="Banco Interamericano de Desarrollo" style="float: right; width: 220px; height: auto;">

# Tamaño de mercado y potencial exportador restante por producto

Editor: Juan Diego Heredia Niño - jheredia@fedesarrollo.org.co

Septiembre 2026

### Importar librerías

La siguiente celda marca el inicio de un nuevo bloque de trabajo independiente:

**Librerías utilizadas:**
- `pandas` y `numpy`: para manipulación y análisis de datos estructurados.
- `funciones.nearest_product`: identificación del producto más cercano en el espacio de producto.

In [1]:
import pandas as pd
import numpy as np
from funciones.nearest_product import nearest_product

### Configuración de rutas

Carga las rutas del proyecto y las opciones globales desde `config.py`, instalado como
paquete editable en la raíz del proyecto.

In [2]:
from config import temp, tables

### Parámetros

Define el umbral de proximidad y el año de referencia del comercio observado.

In [3]:
cercania = 0.65

anio_referencia = 2023

### Clasificación geoeconómica del importador

Replica la clasificación usada en la cadena de limpieza, necesaria para completar el grupo de
los importadores que no aparecen en la base de comercio del año de referencia.

In [4]:
def clasificar_pais(p):
    if p in {'AUT', 'BEL', 'BGR', 'HRV', 'CYP', 'CZE', 'DNK', 'EST', 'FIN', 'FRA', 'DEU',
      'GRC', 'HUN', 'IRL', 'ITA', 'LVA', 'LTU', 'LUX', 'MLT', 'NLD', 'POL', 'PRT',
      'ROM', 'SVK', 'SVN', 'ESP', 'SWE'}:
        return 'Unión Europea'
    elif p in {'GBR'}:
        return 'Reino Unido'
    elif p in {'CHN', 'HKG', 'IDN', 'JPN', 'KOR', 'MYS', 'PHL', 'SGP', 'TWN', 'THA', 'VNM', 'RUS'}:
        return 'APEC Asia'
    elif p in {'USA'}:
        return 'Estados Unidos'
    elif p in {'CAN'}:
        return 'Canadá'
    elif p in {'ARG', 'BHS', 'BLZ', 'BOL', 'BRA', 'CHL', 'COL', 'CRI', 'CUB', 'DMA', 'DOM', 'ECU',
       'GRD', 'GTM', 'GUY', 'HND', 'JAM', 'MEX', 'NIC', 'PAN', 'PER', 'PRY', 'SLV',
       'TTO', 'URY', 'VCT'}:
        return 'América Latina y el Caribe'
    else:
        return 'Otro'

### Lectura de datos

Carga la cartera caracterizada, el potencial exportador, el comercio bilateral, la matriz de
proximidad y los nombres de los productos. Los códigos de potencial con menos de seis dígitos
se marcan como faltantes para tratarlos por separado a cuatro dígitos.

In [5]:
df_products = pd.read_parquet(temp / 'df_products_caracterizado.parquet')
df_products['HS4'] = df_products['HS6'].str[:4]

df_potencial_exportador = pd.read_parquet(temp / 'potencial_exportador.parquet')
df_potencial_exportador.loc[df_potencial_exportador['HS6'].str.len() < 6, 'HS6'] = np.nan

df_comercio = pd.read_parquet(temp / 'df_comercio.parquet')
df_product_space = pd.read_parquet(temp / 'df_product_space.parquet')
df_hs_names = pd.read_parquet(temp / 'df_hs_names.parquet')

### Potencial disponible a seis y a cuatro dígitos

Separa el potencial exportador reportado a seis dígitos del reportado únicamente a cuatro.

**Pasos realizados:**

1. **Potencial a seis dígitos:** marca los productos que cuentan con estimación directa.

2. **Potencial residual a cuatro dígitos:** resta del potencial de cada partida de cuatro dígitos lo que ya está explicado por sus productos de seis, y conserva el remanente.

In [6]:
df_hs6 = df_potencial_exportador.dropna()

df_tienen_potencialHS6 = df_hs6[['HS6']].drop_duplicates()
df_tienen_potencialHS6['Potencial HS6'] = True

df_hs4_total = df_potencial_exportador[df_potencial_exportador['HS6'].isna()][['ISO Exportador', 'ISO Importador', 'HS2', 'HS4','Potencial Exportador']].dropna(subset=['HS4']).drop_duplicates()

df_hs6_agg = df_potencial_exportador.dropna(subset=['HS6']).groupby(['ISO Exportador', 'ISO Importador', 'HS2', 'HS4']).agg({'Potencial Exportador': 'sum'}).reset_index()

df_hs4_restante = df_hs4_total.merge(df_hs6_agg, on=['ISO Exportador', 'ISO Importador', 'HS2', 'HS4'], how='outer', suffixes=(' HS4', ' HS6')).fillna({'Potencial Exportador HS6': 0}).dropna(subset=['Potencial Exportador HS4'])
df_hs4_restante['Potencial Exportador'] = np.where(df_hs4_restante['Potencial Exportador HS4'] <= df_hs4_restante['Potencial Exportador HS6'], df_hs4_restante['Potencial Exportador HS4'], df_hs4_restante['Potencial Exportador HS4'] - df_hs4_restante['Potencial Exportador HS6'])

df_hs4 = df_hs4_restante[['ISO Exportador', 'ISO Importador', 'HS2', 'HS4', 'Potencial Exportador']].drop_duplicates()

df_tienen_potencialHS4 = df_hs4[['HS4']].drop_duplicates()
df_tienen_potencialHS4['Potencial HS4'] = True

### Partición de la cartera según la disponibilidad de potencial

Divide los productos en tres grupos: los que tienen potencial a seis dígitos, los que solo lo
tienen a cuatro, y los que no tienen ninguno y deben aproximarse por cercanía.

In [7]:
df_products = df_products.merge(df_tienen_potencialHS6, on='HS6', how='left')
df_products['Potencial HS6'] = df_products['Potencial HS6'].fillna(False)

df_products_hs4 = df_products.query("`Potencial HS6`== False").merge(df_tienen_potencialHS4, on='HS4', how='left')
df_products_hs4['Potencial HS4'] = df_products_hs4['Potencial HS4'].fillna(False)

df_products_no_potencial = df_products_hs4.query("`Potencial HS4` == False")
df_products_hs4 = df_products_hs4.query("`Potencial HS4` == True")
df_products_hs6 = df_products.query("`Potencial HS6` == True")

df_products_no_potencial.drop(columns=['Potencial HS6', 'Potencial HS4'], inplace=True)

### Potencial exportador restante

Para cada grupo descuenta del potencial el comercio ya observado en el año de referencia y
conserva el remanente. Descarta los destinos que no pertenecen a ninguno de los grupos
geoeconómicos de interés.

Los productos sin potencial propio toman el de su producto más cercano en el espacio de
producto.

In [8]:
df_products_hs4.drop(columns=['Potencial HS6', 'Potencial HS4'], inplace=True)
df_products_hs4 = df_products_hs4.merge(df_hs4, on='HS4')
df_products_hs4 = df_products_hs4.merge(df_comercio.query("Año == @anio_referencia")[['ISO Exportador', 'ISO Importador','Grupo Importador','HS6','HS4','Valor Comercio']], on=['ISO Exportador', 'ISO Importador', 'HS6','HS4'], how='left').reset_index(drop=True)
df_products_hs4['Valor Comercio'] = df_products_hs4['Valor Comercio'].fillna(0)
df_products_hs4['Grupo Importador'] = df_products_hs4['Grupo Importador'].fillna(df_products_hs4['ISO Importador'].apply(clasificar_pais))
df_products_hs4['Potencial Exportador Restante'] = np.where(df_products_hs4['Potencial Exportador'] >= df_products_hs4['Valor Comercio'], df_products_hs4['Potencial Exportador'] - df_products_hs4['Valor Comercio'], 0)
df_products_hs4 = df_products_hs4.query("`Grupo Importador` != 'Otro'").reset_index(drop=True)
print(f" Los producto únicos que se pegaron por HS4 son {len(df_products_hs4['HS6'].unique())}")

df_products_hs6.drop(columns=['Potencial HS6'], inplace=True)
df_products_hs6 = df_products_hs6.merge(df_hs6, on=['HS4','HS6'], how='left')
df_products_hs6 = df_products_hs6.merge(df_comercio.query("Año == @anio_referencia")[['ISO Exportador', 'ISO Importador','Grupo Importador','HS6','Valor Comercio']], on=['ISO Exportador', 'ISO Importador', 'HS6'], how='left').reset_index(drop=True)
df_products_hs6['Valor Comercio'] = df_products_hs6['Valor Comercio'].fillna(0)
df_products_hs6['Grupo Importador'] = df_products_hs6['Grupo Importador'].fillna(df_products_hs6['ISO Importador'].apply(clasificar_pais))
df_products_hs6['Potencial Exportador Restante'] = np.where(df_products_hs6['Potencial Exportador'] >= df_products_hs6['Valor Comercio'], df_products_hs6['Potencial Exportador'] - df_products_hs6['Valor Comercio'], 0)
df_products_hs6 = df_products_hs6.query("`Grupo Importador` != 'Otro'").reset_index(drop=True)
print(f" Los producto únicos que se pegaron por HS6 son {len(df_products_hs6['HS6'].unique())}")

df_products_no_potencial = pd.concat([df_products_no_potencial,df_products_no_potencial[['HS6']].map(lambda x: nearest_product(df_product_space,x,list(set(df_hs6['HS6']) & set(df_product_space.columns)))[0] if x in df_product_space.columns else '').rename(columns={'HS6': 'HS6 Nearest'})], axis=1)
df_products_no_potencial['HS4 Nearest'] = df_products_no_potencial['HS6 Nearest'].str[:4]
df_products_no_potencial = df_products_no_potencial.merge(df_hs6, left_on=['HS4 Nearest','HS6 Nearest'], right_on=['HS4','HS6'], how='left' ,suffixes=('', '_y')).drop(columns=['HS4_y', 'HS6_y','HS2','HS4 Nearest','HS6 Nearest'])
df_products_no_potencial = df_products_no_potencial.merge(df_comercio.query("Año == @anio_referencia")[['ISO Exportador', 'ISO Importador','Grupo Importador','HS6','HS4','Valor Comercio']], on=['ISO Exportador', 'ISO Importador', 'HS6','HS4'], how='left').reset_index(drop=True)
df_products_no_potencial['Valor Comercio'] = df_products_no_potencial['Valor Comercio'].fillna(0)
df_products_no_potencial['Grupo Importador'] = df_products_no_potencial['Grupo Importador'].fillna(df_products_no_potencial['ISO Importador'].apply(clasificar_pais))
df_products_no_potencial['Potencial Exportador Restante'] = np.where(df_products_no_potencial['Potencial Exportador'] >= df_products_no_potencial['Valor Comercio'], df_products_no_potencial['Potencial Exportador'] - df_products_no_potencial['Valor Comercio'], 0)
df_products_no_potencial = df_products_no_potencial.query("`Grupo Importador` != 'Otro'").reset_index(drop=True)
print(f" Los producto únicos que se pegaron por cercanía son {len(df_products_no_potencial['HS6'].unique())}")

 Los producto únicos que se pegaron por HS4 son 53


 Los producto únicos que se pegaron por HS6 son 250


 Los producto únicos que se pegaron por cercanía son 359


### Consolidación por producto

Suma el potencial restante de los tres grupos por producto y le adjunta su descripción.

El notebook original repetía esta misma concatenación en cuatro celdas distintas; aquí se
define una vez como función y se invoca donde corresponde.

In [9]:
def consolidar_potencial():
    return pd.concat([
        df_products_hs4.fillna({'Leamer':'TBD'}).groupby(['HS6', 'Fuente', 'Categoría', 'Margen', 'Leamer', 'Type'])['Potencial Exportador Restante'].sum().reset_index(),
        df_products_hs6.fillna({'Leamer':'TBD'}).groupby(['HS6', 'Fuente', 'Categoría', 'Margen', 'Leamer', 'Type'])['Potencial Exportador Restante'].sum().reset_index(),
        df_products_no_potencial.fillna({'Leamer':'TBD'}).groupby(['HS6', 'Fuente', 'Categoría', 'Margen', 'Leamer', 'Type'])['Potencial Exportador Restante'].sum().reset_index()
    ]).merge(df_hs_names, on='HS6', how='left')


df_products_mercado = consolidar_potencial()

temp.mkdir(parents=True, exist_ok=True)
df_products_mercado.to_parquet(temp / 'df_products_mercado.parquet', index=False)
print(f"productos con tamaño de mercado: {df_products_mercado['HS6'].nunique()}")

productos con tamaño de mercado: 662


### Tabla de potencial exportador por producto

Ordena el resultado por margen y potencial, redondea a tres decimales y expresa el valor con
coma decimal para su lectura en español.

In [10]:
df_output = consolidar_potencial().drop(columns=['Fuente', 'Categoría']).drop_duplicates().sort_values(by=['Margen','Potencial Exportador Restante'], ascending=[True,False]).round(3).reset_index(drop=True)
df_output['Potencial Exportador Restante'] = df_output['Potencial Exportador Restante'].astype(str).str.replace('.', ',')

tables.mkdir(parents=True, exist_ok=True)
df_output.to_excel(tables / 'df_potencial_exportador.xlsx', index=False)